In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import matplotlib.dates as mdates
import seaborn as sns
from FinMind.data import DataLoader
api = DataLoader()

In [2]:
class DCA_With_Dip_Buying:
    """
    定期定額 + 大跌加碼策略
    """
    def __init__(self, base_monthly_investment=10000):
        self.base_monthly = base_monthly_investment
        self.investment_log = []
        self.shares_owned = 0
        self.total_invested = 0
        self.dividend_log = []  # 新增：記錄領到的股利

    def get_monthly_fixed_dates(self, start_date, end_date, day_of_month=15):
        """
        生成每月的固定日期（若遇假日，則使用前一個交易日）
        
        Parameters:
        -----------
        start_date : str or datetime
            開始日期
        end_date : str or datetime
            結束日期
        day_of_month : int
            每月的第幾天（1-31）
        """
        # 生成理論上的每月固定日期
        monthly_dates = pd.date_range(
            start=start_date,
            end=end_date,
            freq=f'MS'  # 每月開始
        )
        
        # 將每個月調整到指定日期
        fixed_dates = []
        for date in monthly_dates:
            # 嘗試創建指定日期的日期
            try:
                target_date = date.replace(day=day_of_month)
            except ValueError:
                # 如果該月沒有指定日期（如2月30日），則使用該月最後一天
                last_day = pd.Timestamp(date.year, date.month, 1) + pd.offsets.MonthEnd(1)
                target_date = last_day
            
            fixed_dates.append(target_date)
        
        return pd.DatetimeIndex(fixed_dates)

    def apply_strategy(self, df, div_df, invest_day=None):
        """
        应用策略，并整合除权息信息。
        
        Parameters:
        -----------
        df : pd.DataFrame
            包含'date'和'close'等列的股价数据
        div_df : pd.DataFrame
            包含'CashExDividendTradingDate'(除息日)和'CashEarningsDistribution'(现金股利)的DataFrame
        invest_day : int
            每月的投资日期
        """
        # 初始化参数和日志（保留你原有的部分）
        if invest_day is None:
            invest_day = self.invest_day
        self.monthly_date = self.get_monthly_fixed_dates(
            start_date=df['date'].iloc[0],
            end_date=df['date'].iloc[-1],
            day_of_month=invest_day
        )
        
        dip_thresholds = {'small_dip': 0.03, 'medium_dip': 0.05, 'big_dip': 0.08}
        dip_multipliers = {'small_dip': 1.5, 'medium_dip': 2, 'big_dip': 5}
        
        # 初始化一个列表来记录每次除息时获得的现金股利
        self.dividend_log = []
        
        for i in range(1, len(df)):
            current_date = df['date'].iloc[i]
            current_price = df['close'].iloc[i]
            prev_price = df['close'].iloc[i-1]
            
            daily_return = (current_price - prev_price) / prev_price
            is_monthly_day = current_date in self.monthly_date
            investment_amount = 0
            trigger = None
            
            # 判断是否是除息日[citation:1]
            # 注意：这里假设div_df中的日期与df中的日期格式一致
            if not div_df.empty and current_date in div_df['CashExDividendTradingDate'].values:
                # 获取该除息日的现金股利金额
                cash_dividend = div_df.loc[
                    div_df['CashExDividendTradingDate'] == current_date, 
                    'CashEarningsDistribution'
                ].iloc[0]
                
                # 计算当前持有的股份应得的现金股利
                # 除息日当天持有的股份有权领取股利[citation:4]
                if self.shares_owned > 0:
                    dividend_income = self.shares_owned * cash_dividend
                    self.dividend_log.append({
                        'date': current_date,
                        'dividend_per_share': cash_dividend,
                        'dividend_income': dividend_income,
                        'shares_owned': self.shares_owned
                    })
            
            # 原有的投资触发逻辑（保留你原有的部分）
            if is_monthly_day:
                investment_amount += self.base_monthly
                trigger = "定期定额"
            if daily_return < -dip_thresholds['big_dip']:
                investment_amount += self.base_monthly * dip_multipliers['big_dip']
                trigger = "大跌加码(>8%)"
            elif daily_return < -dip_thresholds['medium_dip']:
                investment_amount += self.base_monthly * dip_multipliers['medium_dip']
                trigger = "中跌加码(>5%)"
            elif daily_return < -dip_thresholds['small_dip']:
                investment_amount += self.base_monthly * dip_multipliers['small_dip']
                trigger = "小跌加码(>3%)"
                
            # 执行投资
            if investment_amount > 0:
                shares_bought = investment_amount / current_price
                self.shares_owned += shares_bought
                self.total_invested += investment_amount
                
                self.investment_log.append({
                    'date': current_date,
                    'price': current_price,
                    'amount': investment_amount,
                    'shares': shares_bought,
                    'trigger': trigger,
                    'total_shares': self.shares_owned  # 记录累计持股
                })
        
        # 返回投资记录和股利记录
        investment_df = pd.DataFrame(self.investment_log)
        dividend_df = pd.DataFrame(self.dividend_log) if self.dividend_log else pd.DataFrame()
        return investment_df, dividend_df

    
    def get_performance(self, current_price):
        """
        计算绩效，包含现金股利收益。
        
        Returns:
        --------
        tuple: (current_value, total_return, return_rate, total_dividend)
        """
        if self.shares_owned == 0:
            return 0, 0, 0, 0
        
        # 计算股票市值
        current_value = self.shares_owned * current_price
        
        # 计算累计现金股利总额
        total_dividend = sum([log['dividend_income'] for log in self.dividend_log]) if hasattr(self, 'dividend_log') else 0
        
        # 计算总回报
        # 总回报 = 当前股票市值 + 累计收到的现金股利 - 总投入本金
        total_return = (current_value + total_dividend) - self.total_invested
        
        # 计算总回报率
        if self.total_invested > 0:
            return_rate = (total_return / self.total_invested) * 100
        else:
            return_rate = 0
        
        return current_value, total_return, return_rate, total_dividend

In [3]:
today = datetime.today() #datetime.strptime('2025-04-30', "%Y-%m-%d")
start_date = (today - relativedelta(years=2)).strftime("%Y-%m-%d")
end_date = today.strftime("%Y-%m-%d")
print('start date', start_date)
print('end date', end_date)

ticker = '00830'
df = api.taiwan_stock_daily(
    stock_id=ticker,
    start_date=start_date,
    end_date=end_date
)

div_df = api.taiwan_stock_dividend(
    stock_id=ticker,
    start_date=start_date,
    end_date=end_date
)
div_df = div_df[['CashExDividendTradingDate', 'CashEarningsDistribution']]

2026-02-03 17:25:15.361 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 00830


start date 2024-02-03
end date 2026-02-03


2026-02-03 17:25:16.082 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockDividend, data_id: 00830


In [4]:
# 執行策略
strategy = DCA_With_Dip_Buying(base_monthly_investment=8000)
investment_df, dividend_df = strategy.apply_strategy(df, div_df, 9)

current_price = df['close'].iloc[-1]
current_value, total_return, return_rate, total_dividend = strategy.get_performance(current_price)

print(f"總投入金額: {strategy.total_invested:,.0f}元")
print(f"累计现金股利: {total_dividend:.2f} 元")
print(f"目前持有市值: {current_value:,.0f}元")
print(f"總報酬: {total_return:,.0f}元")
print(f"報酬率: {return_rate:.2f}%")
print(f"持有股數: {strategy.shares_owned:.2f}股")

# 显示投资记录
print("\n=== 最近10次投资记录 ===")
print(investment_df.tail(100) if len(investment_df) > 10 else investment_df)

# 显示股利记录（如果存在）
if not dividend_df.empty:
    print("\n=== 现金股利记录 ===")
    print(dividend_df)
else:
    print("\n=== 现金股利记录 ===")
    print("期间内无现金股利发放")

總投入金額: 628,000元
累计现金股利: 150641.18 元
目前持有市值: 852,530元
總報酬: 375,171元
報酬率: 59.74%
持有股數: 14826.61股

=== 最近10次投资记录 ===
          date  price   amount       shares    trigger  total_shares
0   2024-03-11  43.62  12000.0   275.103164  小跌加码(>3%)    275.103164
1   2024-04-09  43.74   8000.0   182.898948       定期定额    458.002112
2   2024-04-19  40.44  12000.0   296.735905  小跌加码(>3%)    754.738017
3   2024-05-02  41.85  12000.0   286.738351  小跌加码(>3%)   1041.476368
4   2024-05-09  43.40   8000.0   184.331797       定期定额   1225.808166
5   2024-06-21  51.10  12000.0   234.833659  小跌加码(>3%)   1460.641825
6   2024-07-09  53.05   8000.0   150.801131       定期定额   1611.442956
7   2024-07-12  52.00  12000.0   230.769231  小跌加码(>3%)   1842.212187
8   2024-07-18  50.25  16000.0   318.407960  中跌加码(>5%)   2160.620147
9   2024-07-26  47.29  16000.0   338.337915  中跌加码(>5%)   2498.958062
10  2024-08-02  45.45  16000.0   352.035204  中跌加码(>5%)   2850.993266
11  2024-08-05  40.60  40000.0   985.221675  大跌加码(>8%)   3

In [5]:
investment_df

,date,price,amount,shares,trigger,total_shares
0,2024-03-11,43.62,12000.0,275.103164,小跌加码(>3%),275.103164
1,2024-04-09,43.74,8000.0,182.898948,定期定额,458.002112
2,2024-04-19,40.44,12000.0,296.735905,小跌加码(>3%),754.738017
3,2024-05-02,41.85,12000.0,286.738351,小跌加码(>3%),1041.476368
4,2024-05-09,43.40,8000.0,184.331797,定期定额,1225.808166
5,2024-06-21,51.10,12000.0,234.833659,小跌加码(>3%),1460.641825
6,2024-07-09,53.05,8000.0,150.801131,定期定额,1611.442956
7,2024-07-12,52.00,12000.0,230.769231,小跌加码(>3%),1842.212187
8,2024-07-18,50.25,16000.0,318.407960,中跌加码(>5%),2160.620147
9,2024-07-26,47.29,16000.0,338.337915,中跌加码(>5%),2498.958062
